In [43]:
import pandas as pd
import numpy as np

In [44]:
df = pd.read_csv("../data/process/telco_customer_churn_cleaned.csv")

print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [45]:
def categorize_tenure(tenure):
    if tenure <= 12:
        return "0-12 Months"
    elif tenure <= 24:
        return "13-24 Months"
    elif tenure <= 48:
        return "25-48 Months"
    else:
        return "49-72 Months"

df["tenure_group"] = df["tenure"].apply(categorize_tenure)

In [46]:
df[["tenure", "tenure_group"]].head(10)

,tenure,tenure_group
0,1,0-12 Months
1,34,25-48 Months
2,2,0-12 Months
3,45,25-48 Months
4,2,0-12 Months
5,8,0-12 Months
6,22,13-24 Months
7,10,0-12 Months
8,28,25-48 Months
9,62,49-72 Months


In [47]:
df["is_month_to_month"] = (
    df["Contract"] == "Month-to-month"
).astype(int)

In [48]:
df["avg_monthly_spend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    0
)

In [49]:
df[
    [
        "tenure",
        "tenure_group",
        "Contract",
        "is_month_to_month",
        "TotalCharges",
        "avg_monthly_spend"
    ]
].head(10)

,tenure,tenure_group,Contract,is_month_to_month,TotalCharges,avg_monthly_spend
0,1,0-12 Months,Month-to-month,1,29.85,29.850000
1,34,25-48 Months,One year,0,1889.50,55.573529
2,2,0-12 Months,Month-to-month,1,108.15,54.075000
3,45,25-48 Months,One year,0,1840.75,40.905556
4,2,0-12 Months,Month-to-month,1,151.65,75.825000
5,8,0-12 Months,Month-to-month,1,820.50,102.562500
6,22,13-24 Months,Month-to-month,1,1949.40,88.609091
7,10,0-12 Months,Month-to-month,1,301.90,30.190000
8,28,25-48 Months,Month-to-month,1,3046.05,108.787500
9,62,49-72 Months,One year,0,3487.95,56.257258


In [50]:
df[[
    "tenure_group",
    "is_month_to_month",
    "avg_monthly_spend"
]].isnull().sum()

tenure_group         0
is_month_to_month    0
avg_monthly_spend    0
dtype: int64

In [51]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customerID         7043 non-null   str    
 1   gender             7043 non-null   str    
 2   SeniorCitizen      7043 non-null   int64  
 3   Partner            7043 non-null   str    
 4   Dependents         7043 non-null   str    
 5   tenure             7043 non-null   int64  
 6   PhoneService       7043 non-null   str    
 7   MultipleLines      7043 non-null   str    
 8   InternetService    7043 non-null   str    
 9   OnlineSecurity     7043 non-null   str    
 10  OnlineBackup       7043 non-null   str    
 11  DeviceProtection   7043 non-null   str    
 12  TechSupport        7043 non-null   str    
 13  StreamingTV        7043 non-null   str    
 14  StreamingMovies    7043 non-null   str    
 15  Contract           7043 non-null   str    
 16  PaperlessBilling   7043 non-null   

In [52]:
df_encoded = pd.get_dummies(
    df,
    columns=[
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod"
    ],
    drop_first=True,
    dtype=int
)

In [53]:
X = df_encoded.drop(
    columns=["Churn", "customerID", "tenure_group"],
    errors="ignore"
)

y = df_encoded["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [54]:
print(X.select_dtypes(exclude="number").columns)

Index([], dtype='str')


In [55]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nMissing values:")
print(X.isnull().sum().sum())

X shape: (7043, 32)
y shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Missing values:
0


In [56]:
from sklearn.model_selection import train_test_split

In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [58]:
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training data: (5634, 32)
Testing data: (1409, 32)

Training target distribution:
Churn
0    73.464679
1    26.535321
Name: proportion, dtype: float64

Testing target distribution:
Churn
0    73.456352
1    26.543648
Name: proportion, dtype: float64


In [59]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=2000)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [60]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8076650106458482
